<div style="background:linear-gradient(135deg,#0f172a,#1e3a8a,#312e81);padding:28px;border-radius:18px;color:white;border:1px solid #334155">
  <h1 style="margin:0;font-size:34px">AIC2026 Backend ↔ Frontend Handoff Guide</h1>
  <p style="font-size:18px;margin:8px 0 0;color:#dbeafe">Tài liệu bàn giao cho team Backend / AI / Database để nối vào frontend Retrieval System</p>
  <p style="font-size:14px;margin-top:14px;color:#bfdbfe">Notebook này không dạy code backend chi tiết. Nó chỉ chỉ rõ: backend cần làm phần nào, source backend nên đặt ở đâu, frontend đang chờ API nào, file frontend nào cần sửa khi nối thật.</p>
</div>

<div style="background:#fff7ed;border-left:6px solid #f97316;padding:16px;border-radius:10px;color:#431407">
<b>Đặt file notebook này ở đâu?</b><br>
Nên đặt notebook này ở <b>root project</b>, ngang cấp với <code>package.json</code>, <code>src/</code>, <code>vite.config.ts</code>. Khi đó các link như <a href="src/api.ts">src/api.ts</a> có thể bấm để mở trực tiếp file trong nhiều môi trường như VS Code / JupyterLab.
</div>

## 0. Mục tiêu của tài liệu này

Frontend hiện tại là một prototype cho hệ thống truy xuất multimedia của AIC2026. Người dùng nhập mô tả hoặc câu hỏi, chọn mode như KIS / QA / TRAKE, chọn Top K, model, collection, sau đó giao diện hiển thị các frame liên quan. Người dùng có thể preview frame, đưa frame sang Answer Builder/Recheck và export CSV để nộp.

Tài liệu này dành cho các bạn phụ trách backend/AI/database. Các bạn không cần sửa giao diện nhiều. Các bạn cần hiểu frontend đang cần gì để nối các phần sau:

<div style="display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin-top:16px">
  <div style="background:#0f172a;color:white;padding:14px;border-radius:12px;border:1px solid #334155"><b>AI Models</b><br><span style="color:#cbd5e1">CLIP / SigLIP / SigLIP2 / YOLO / OCR / ASR / VLM</span></div>
  <div style="background:#052e16;color:white;padding:14px;border-radius:12px;border:1px solid #16a34a"><b>Vector DB</b><br><span style="color:#bbf7d0">Qdrant / pgvector / Supabase / FAISS</span></div>
  <div style="background:#1e1b4b;color:white;padding:14px;border-radius:12px;border:1px solid #6366f1"><b>API</b><br><span style="color:#c7d2fe">/api/search, /api/media, /api/models</span></div>
  <div style="background:#450a0a;color:white;padding:14px;border-radius:12px;border:1px solid #ef4444"><b>Runtime</b><br><span style="color:#fecaca">localhost / LAN / deploy nội bộ</span></div>
</div>

## 1. Bản đồ source frontend hiện tại

| Khu vực | File cần biết | Vai trò | Backend cần quan tâm gì? |
|---|---|---|---|
| Root render | [src/main.tsx](src/main.tsx) | Gắn React app vào `index.html` | Không cần sửa |
| Layout chính | [src/App.tsx](src/App.tsx) | Chia giao diện 3 cột: Search - Results - Recheck | Không cần sửa trừ logo/title/layout |
| State toàn app | [src/store.ts](src/store.ts) | Lưu query, mode, results, recheck list, preview, loading | Cần biết dữ liệu result phải khớp type trong đây |
| API adapter | [src/api.ts](src/api.ts) | Hiện đang mock search. Đây là điểm chính để nối backend thật | Backend thống nhất endpoint với file này |
| Search UI | [src/components/SearchPanel.tsx](src/components/SearchPanel.tsx) | Ô query, mode, topK, collection, model, filter | Backend muốn nhận thêm filter thì báo frontend sửa payload ở đây |
| Grid kết quả | [src/components/ResultGrid.tsx](src/components/ResultGrid.tsx) | Hiển thị danh sách kết quả | Cần trả `results` dạng list |
| Card frame | [src/components/ResultCard.tsx](src/components/ResultCard.tsx) | Mỗi frame: ảnh, score, videoId, frameId, preview, add recheck | Cần trả link ảnh/metadata đúng |
| Preview lớn | [src/components/PreviewModal.tsx](src/components/PreviewModal.tsx) | Xem frame to, caption/OCR/ASR/object, add recheck | Cần trả metadata càng đầy đủ càng tốt |
| Answer Builder | [src/components/RecheckPanel.tsx](src/components/RecheckPanel.tsx) | Danh sách candidate và export CSV | Backend thường không cần can thiệp |
| Media URL helper | [src/utils/media.ts](src/utils/media.ts) | Tạo URL ảnh/video từ path backend trả | Cần thống nhất đường dẫn media |
| Mock data | [src/data/mockResults.ts](src/data/mockResults.ts) | Dữ liệu giả để frontend chạy khi chưa có backend | Backend nối thật thì không dùng nữa |
| Vite config | [vite.config.ts](vite.config.ts) | Config dev server/proxy | Có thể cần sửa proxy tới backend |
| Package | [package.json](package.json) | Scripts chạy frontend | Có thể thêm script chạy host LAN |

## 2. Luồng hoạt động hiện tại của frontend

```text
Người dùng nhập query ở SearchPanel
        ↓
SearchPanel gọi api.search(query, collection, model, topK)
        ↓
src/api.ts hiện tại trả mock results
        ↓
store.ts lưu searchResults
        ↓
ResultGrid đọc searchResults và render ResultCard
        ↓
Người dùng bấm preview hoặc add vào Recheck
        ↓
PreviewModal / RecheckPanel đọc dữ liệu từ store
        ↓
RecheckPanel export CSV theo mode KIS / QA / TRAKE
```

<div style="background:#ecfeff;border-left:6px solid #06b6d4;padding:14px;border-radius:10px;color:#164e63">
<b>Điểm nối backend quan trọng nhất:</b> <a href="src/api.ts"><code>src/api.ts</code></a>. Khi backend sẵn sàng, frontend chỉ cần thay mock search trong file này bằng request thật tới backend.
</div>

## 3. Nguyên tắc chia việc giữa frontend và backend

| Việc | Thuộc frontend | Thuộc backend/AI/database |
|---|---:|---:|
| Giao diện 3 cột, nút, modal, màu sắc | ✅ | ❌ |
| Nhập query, chọn mode, chọn Top K | ✅ | ❌ |
| Export CSV KIS/QA/TRAKE từ Answer Builder | ✅ | ❌ |
| Chạy CLIP / SigLIP / SigLIP2 | ❌ | ✅ |
| Chạy YOLO object detection | ❌ | ✅ |
| OCR / ASR / caption / metadata extraction | ❌ | ✅ |
| Lưu vector embedding, metadata, index | ❌ | ✅ |
| API search trả kết quả về frontend | ❌ | ✅ |
| Serve ảnh/video frame để frontend hiển thị | ❌ | ✅ |
| Cache kết quả, Redis, ranking, reranking | ❌ | ✅ |

<div style="background:#fef2f2;border-left:6px solid #ef4444;padding:14px;border-radius:10px;color:#7f1d1d">
<b>Không nên làm:</b> Không nhét Python model, database driver, YOLO, CLIP trực tiếp vào thư mục <code>src/</code> của frontend. <code>src/</code> là React/TypeScript chạy trên browser, không phù hợp chạy model AI nặng.
</div>

## 4. Backend source code nên đặt ở đâu?

Có 2 cách tổ chức. Team chọn 1 cách, nhưng phải báo frontend biết endpoint cuối cùng.

### Cách A — Cùng repo với frontend

```text
project-root/
  src/                  ← frontend React, không để model Python ở đây
  backend/              ← backend Python/FastAPI/AI/database đặt ở đây
  package.json
  vite.config.ts
```

Backend copy source code AI/API/database vào thư mục <code>backend/</code>. Frontend vẫn chạy bằng <code>npm run dev</code>. Backend chạy bằng lệnh riêng, ví dụ port <code>8000</code>.

### Cách B — Tách repo frontend và backend

```text
frontend-repo/
  src/
  package.json

backend-repo/
  app/
  models/
  services/
  database/
```

Cách này sạch hơn nếu backend nặng. Frontend chỉ cần biết URL backend, ví dụ <code>http://localhost:8000</code> hoặc IP LAN của máy host.

<div style="background:#f0fdf4;border-left:6px solid #22c55e;padding:14px;border-radius:10px;color:#14532d">
<b>Khuyến nghị cho team:</b> Nếu mới prototype, dùng Cách A cho dễ bàn giao. Nếu backend bắt đầu có model nặng, data lớn, docker, GPU thì chuyển sang Cách B.
</div>

## 5. Backend cần cung cấp những API nào?

Frontend không quan tâm backend dùng CLIP, SigLIP, YOLO, Redis, Qdrant hay Supabase ở bên trong. Frontend chỉ cần API trả đúng format.

| API | Mục đích | Frontend gọi từ đâu | Trạng thái nên có |
|---|---|---|---|
| `GET /api/health` | Kiểm tra backend sống chưa | Có thể thêm vào debug | Nên có |
| `GET /api/models` | Lấy danh sách model như CLIP/SigLIP | [src/api.ts](src/api.ts), [SearchPanel](src/components/SearchPanel.tsx) | Nên có |
| `GET /api/collections` | Lấy danh sách collection/index | [src/api.ts](src/api.ts), [SearchPanel](src/components/SearchPanel.tsx) | Nên có |
| `POST /api/search` hoặc `/api/search/` | Search chính theo query | [src/api.ts](src/api.ts) | Bắt buộc |
| `POST /api/search/continuous` | Tìm tương tự từ một frame | [ResultCard](src/components/ResultCard.tsx), [PreviewModal](src/components/PreviewModal.tsx) | Có thể làm sau |
| `GET /api/media/image/...` | Serve ảnh/keyframe | [src/utils/media.ts](src/utils/media.ts) | Bắt buộc nếu ảnh không public |
| `GET /api/media/video/...` | Serve video gốc/clip preview | [VideoModal](src/components/VideoModal.tsx) | Có thể làm sau |

<div style="background:#eff6ff;border-left:6px solid #3b82f6;padding:14px;border-radius:10px;color:#1e3a8a">
<b>File frontend cần sửa khi API thật có:</b> gần như chỉ cần sửa <a href="src/api.ts"><code>src/api.ts</code></a> và có thể sửa <a href="src/utils/media.ts"><code>src/utils/media.ts</code></a> nếu đường dẫn ảnh/video thay đổi.
</div>

## 6. Search request frontend sẽ gửi cho backend

Khi người dùng bấm Search, backend nên nhận một request có tinh thần như sau:

```json
{
  "query": "một người đang đi qua đường dưới trời mưa",
  "mode": "KIS",
  "collection": "AIC2026_SIGLIP",
  "model_id": "siglip-so400m",
  "top_k": 100,
  "language": "AUTO",
  "filters": {
    "min_score": null,
    "video_id": null,
    "frame_id": null,
    "objects": [],
    "min_people": null,
    "max_people": null,
    "ocr_contains": null,
    "asr_contains": null
  }
}
```

Trong frontend hiện tại, phần filter UI có thể mới là giao diện. Khi backend cần filter thật, frontend sẽ lấy value từ <a href="src/components/SearchPanel.tsx"><code>SearchPanel.tsx</code></a> rồi đưa vào payload trong <a href="src/api.ts"><code>api.ts</code></a>.

| Field | Ý nghĩa | Backend dùng thế nào? |
|---|---|---|
| `query` | Câu người dùng nhập | Embed bằng CLIP/SigLIP hoặc đưa qua LLM/VLM để phân tích |
| `mode` | AUTO/KIS/QA/TRAKE | Chọn chiến lược search/rerank/answer |
| `collection` | Index/dataset muốn search | Chọn collection trong vector DB |
| `model_id` | Model embedding/ranking | Chọn encoder tương ứng |
| `top_k` | Số lượng kết quả | Giới hạn output trả về |
| `language` | AUTO/VI/EN | Normalize/query translation nếu cần |
| `filters` | Bộ lọc metadata | Filter video/frame/object/OCR/ASR/people |

## 7. Search response backend phải trả về cho frontend

Frontend cần response dạng danh sách `results`. Mỗi result đại diện cho 1 frame/keyframe hoặc 1 candidate frame.

```json
{
  "results": [
    {
      "id": "L21_V041_83991",
      "score": 0.8412,
      "image_path": "L21_V041/83991.jpg",
      "video_id": "L21_V041",
      "timestamp": 1878837,
      "payload": {
        "video_id": "L21_V041",
        "frame_id": "83991",
        "img_link": "L21_V041/83991.jpg",
        "timestamp": 1878837,
        "caption": "A canyon with orange rock walls and sunlight.",
        "ocr": "",
        "asr": "",
        "objects": ["rock", "canyon", "light"],
        "people_count": 0,
        "model": "siglip-so400m"
      }
    }
  ]
}
```

### Field bắt buộc

| Field | Bắt buộc? | Ghi chú |
|---|---:|---|
| `id` | Nên có | Unique id cho card/recheck |
| `score` | Nên có | Điểm similarity/ranking để hiển thị |
| `payload.video_id` | Bắt buộc | Dùng để export CSV |
| `payload.frame_id` | Bắt buộc | Dùng để export CSV |
| `payload.img_link` hoặc `image_path` | Bắt buộc để hiện ảnh | Được đưa qua `getImageUrl()` trong `src/utils/media.ts` |
| `payload.timestamp` | Nên có | Dùng preview/video temporal |
| `payload.caption`, `ocr`, `asr`, `objects` | Không bắt buộc nhưng nên có | Dùng hiển thị metadata và filter |

<div style="background:#fefce8;border-left:6px solid #eab308;padding:14px;border-radius:10px;color:#713f12">
<b>Quan trọng:</b> CSV nộp bài lấy từ <code>payload.video_id</code> và <code>payload.frame_id</code>. Nếu backend trả sai 2 field này, giao diện vẫn có thể hiện ảnh nhưng file nộp sẽ sai.
</div>

## 8. Mapping giữa 3 dạng truy vấn và backend xử lý

| Mode trên giao diện | Tên dạng | Frontend làm gì? | Backend/AI nên làm gì? | CSV export |
|---|---|---|---|---|
| `KIS` | Known Item Search | Người dùng chọn 1 hoặc nhiều frame candidate | Search frame gần nhất theo query, có thể rerank | `video_id,frame_id` |
| `QA` | Video Question Answering | Cho nhập answer trong Answer Builder | Tìm frame/video liên quan, sinh hoặc gợi ý answer | `video_id,frame_id,answer` |
| `TRAKE` | Temporal Retrieval & Alignment of Key Events | Người dùng sắp xếp nhiều frame theo thứ tự | Tách query thành events, search từng event, căn chỉnh thời gian | `video_id,frame1,frame2,...` |
| `AUTO` | Tự đoán mode | UI để tiện test | Backend có thể tự phân loại query hoặc default KIS | Theo mode backend/frontend quyết định |

### Gợi ý backend cho TRAKE

Frontend hiện tại có thể nhận nhiều frame bình thường. Backend có thể làm đơn giản trước: trả nhiều frame đã sort theo timestamp. Khi user đưa vào Answer Builder, user có thể sắp xếp bằng nút up/down rồi export.

Nếu muốn nâng cấp sau, backend có thể trả thêm thông tin `event_index`, `event_text`, `sequence_id`, nhưng frontend cần được báo để render đẹp hơn.

## 9. AI/model code nên nối vào backend ở lớp nào?

Backend nên tách thành các lớp logic. Đây là hướng tổ chức, không bắt buộc đúng tên folder.

```text
backend/
  app.py hoặc main.py          ← entrypoint API server
  routes/                      ← nơi định nghĩa API /search, /media, /models
  services/
    search_service             ← nhận query, gọi embedding, vector DB, rerank
    media_service              ← trả ảnh/video
    answer_service             ← QA / RAG / VLM answer nếu có
  models/
    clip_or_siglip             ← load model embedding
    yolo_detector              ← object detection nếu team dùng
    ocr_asr                    ← OCR/ASR nếu team dùng
  database/
    vector_store               ← Qdrant/Supabase/FAISS wrapper
    metadata_store             ← SQLite/Postgres/json/parquet metadata
  scripts/
    build_index                ← chạy offline để index video/frame
    extract_frames             ← cắt frame/keyframe từ video
```

<div style="background:#f8fafc;border:1px solid #cbd5e1;padding:14px;border-radius:10px;color:#0f172a">
<b>Nguyên tắc:</b> Frontend chỉ gọi API. Frontend không gọi trực tiếp model AI, không gọi trực tiếp vector DB, không giữ API key database.
</div>

## 10. Pipeline offline mà backend cần chuẩn bị

Đây là phần chạy trước khi thi hoặc trước khi search realtime. Nó không liên quan trực tiếp frontend, nhưng quyết định chất lượng retrieval.

```text
Video dataset
  ↓
Extract keyframes / frames
  ↓
Tạo metadata: video_id, frame_id, timestamp, path
  ↓
Tạo embedding ảnh bằng CLIP/SigLIP/SigLIP2
  ↓
Tùy chọn: YOLO object detection
  ↓
Tùy chọn: OCR text từ frame
  ↓
Tùy chọn: ASR transcript từ audio/video
  ↓
Lưu vector vào vector DB
  ↓
Lưu metadata vào DB/file
  ↓
API search đọc index này để trả kết quả cho frontend
```

| Output offline | Frontend sẽ dùng ở đâu? |
|---|---|
| `img_link` / `image_path` | Hiện thumbnail ở ResultCard và ảnh lớn ở PreviewModal |
| `video_id` | Hiển thị, export CSV |
| `frame_id` | Hiển thị, export CSV |
| `timestamp` | Hiển thị, mở video đúng thời điểm nếu có |
| `caption` | Preview overview, hỗ trợ user quyết định |
| `ocr` | Preview tab OCR, filter OCR |
| `asr` | Preview metadata, filter ASR |
| `objects` | Badge objects, filter YOLO |
| `people_count` | Filter people counting |
| `score` | Hiển thị độ liên quan |

## 11. Database/vector store: backend tự chọn, frontend không phụ thuộc

| Lựa chọn | Khi nào dùng? | Frontend cần biết gì? |
|---|---|---|
| Qdrant local | Prototype local, vector search nhanh, dễ docker | Không cần biết, chỉ nhận API results |
| Supabase + pgvector | Muốn cloud/local Postgres, dễ quản trị metadata | Không cần biết, chỉ nhận API results |
| FAISS local | Muốn nhanh, đơn giản, không cần server DB | Không cần biết, backend tự wrap qua API |
| Redis | Cache query/result/session, không phải vector DB chính nếu không cấu hình vector | Không cần biết |
| SQLite/Parquet/JSON | Lưu metadata offline nhỏ/vừa | Không cần biết |

<div style="background:#f0f9ff;border-left:6px solid #0284c7;padding:14px;border-radius:10px;color:#0c4a6e">
<b>Contract quan trọng hơn công nghệ:</b> Backend dùng database gì cũng được, miễn endpoint <code>/api/search</code> trả ra đúng danh sách result mà frontend cần.
</div>

## 12. Ảnh và video: thống nhất đường dẫn với frontend

Frontend không nên nhận ảnh dạng base64 vì nặng. Backend nên trả đường dẫn ảnh, ví dụ:

```text
payload.img_link = "L21_V041/83991.jpg"
```

Sau đó frontend dùng <a href="src/utils/media.ts"><code>src/utils/media.ts</code></a> để biến thành URL thật, ví dụ:

```text
/api/media/image/L21_V041/83991.jpg
```

### Khi backend thay đổi media path

Chỉ cần báo frontend sửa file:

- [src/utils/media.ts](src/utils/media.ts): đổi cách tạo URL ảnh/video.
- Có thể sửa thêm [src/api.ts](src/api.ts) nếu backend trả field khác tên.

### Backend nên đảm bảo

| Yêu cầu | Lý do |
|---|---|
| URL ảnh mở được trực tiếp trên browser | Debug nhanh |
| Ảnh thumbnail không quá nặng | Grid mượt, không lag |
| Có CORS/proxy đúng | Frontend load được ảnh |
| `video_id` và `frame_id` khớp file nộp | CSV không sai |

## 13. Localhost và LAN host cho team test chung

### Chạy local trên một máy

```text
Frontend: http://localhost:5173
Backend:  http://localhost:8000
```

Frontend sẽ gọi backend qua proxy hoặc URL trực tiếp. Nếu dùng Vite proxy, file cần sửa là [vite.config.ts](vite.config.ts).

### Cho cả team cùng Wi-Fi vào chung

Máy host cần expose cả frontend và backend ra LAN:

```text
Frontend: http://IP_MAY_HOST:5173
Backend:  http://IP_MAY_HOST:8000
```

Ví dụ:

```text
http://192.168.1.25:5173
http://192.168.1.25:8000
```

| Bên | Cần làm |
|---|---|
| Frontend | Vite chạy `host: 0.0.0.0` trong [vite.config.ts](vite.config.ts) hoặc script `vite --host 0.0.0.0` |
| Backend | API server chạy host `0.0.0.0`, không chỉ `127.0.0.1` |
| Cả team | Cùng Wi-Fi/LAN, firewall cho phép Node/Python nhận kết nối |

<div style="background:#fff7ed;border-left:6px solid #f97316;padding:14px;border-radius:10px;color:#7c2d12">
<b>Lỗi thường gặp:</b> Frontend máy khác vào được nhưng search lỗi vì frontend vẫn gọi <code>localhost:8000</code>. Với người dùng khác, <code>localhost</code> là máy của họ, không phải máy host. Khi test LAN, cần dùng proxy hoặc API base URL là IP máy host.
</div>

## 14. Cấu hình API base URL: backend cần thống nhất với frontend

Có 2 kiểu nối.

### Kiểu 1 — Dùng Vite proxy

Frontend gọi:

```text
/api/search
/api/media/image/...
```

Vite chuyển tiếp sang backend:

```text
http://localhost:8000/api/search
```

File liên quan: [vite.config.ts](vite.config.ts), [src/api.ts](src/api.ts), [src/utils/media.ts](src/utils/media.ts)

### Kiểu 2 — Gọi trực tiếp backend URL

Frontend gọi trực tiếp:

```text
http://localhost:8000/api/search
```

hoặc khi LAN:

```text
http://192.168.1.25:8000/api/search
```

Kiểu này cần backend bật CORS đúng. File liên quan: [src/api.ts](src/api.ts), có thể thêm biến môi trường nếu team muốn.

<div style="background:#ecfdf5;border-left:6px solid #10b981;padding:14px;border-radius:10px;color:#064e3b">
<b>Khuyến nghị prototype:</b> Dùng Vite proxy để frontend code luôn gọi <code>/api/...</code>. Như vậy khi đổi backend host/port chỉ sửa <code>vite.config.ts</code>, ít đụng component.
</div>

## 15. File frontend nào backend thường yêu cầu sửa?

| Khi backend thay đổi... | File frontend cần sửa | Sửa ý gì? |
|---|---|---|
| Endpoint search khác tên | [src/api.ts](src/api.ts) | Đổi URL `/api/search` |
| Request cần thêm field | [src/api.ts](src/api.ts), [SearchPanel.tsx](src/components/SearchPanel.tsx) | Thêm field vào payload |
| Response trả field khác | [src/api.ts](src/api.ts) | Map response backend về format frontend |
| Ảnh/video nằm URL khác | [src/utils/media.ts](src/utils/media.ts) | Đổi hàm tạo URL media |
| Thêm model/collection động | [src/api.ts](src/api.ts), [SearchPanel.tsx](src/components/SearchPanel.tsx) | Gọi `/api/models`, `/api/collections` |
| Thêm filter thật | [SearchPanel.tsx](src/components/SearchPanel.tsx), [src/api.ts](src/api.ts) | Lấy input filter và gửi backend |
| Thêm ranking fields | [ResultCard.tsx](src/components/ResultCard.tsx), [PreviewModal.tsx](src/components/PreviewModal.tsx) | Hiển thị thêm score/metadata |
| Thêm mode mới | [src/store.ts](src/store.ts), [SearchPanel.tsx](src/components/SearchPanel.tsx), [RecheckPanel.tsx](src/components/RecheckPanel.tsx) | Thêm state, UI, export nếu cần |

<div style="background:#fef2f2;border-left:6px solid #ef4444;padding:14px;border-radius:10px;color:#7f1d1d">
<b>Không nên sửa lung tung:</b> Backend team nên yêu cầu frontend sửa qua các file adapter như <code>api.ts</code>, <code>media.ts</code>, hạn chế sửa component UI nếu không cần.
</div>

## 16. Chuẩn output cho Answer Builder và CSV

Backend không trực tiếp nộp CSV. Frontend RecheckPanel export CSV dựa trên item user đã chọn. Nhưng backend phải trả đúng metadata để CSV đúng.

### KIS

Frontend cần:

```text
payload.video_id
payload.frame_id
```

CSV:

```text
L00_V000,1234
L00_V055,5555
```

### QA

Frontend cần:

```text
payload.video_id
payload.frame_id
answer do user nhập hoặc backend gợi ý
```

Nếu backend có gợi ý answer, có thể trả thêm:

```text
payload.qa_answer
payload.answer
```

Frontend sau này có thể auto-fill vào input answer.

### TRAKE

Frontend cần nhiều frame candidate cùng video hoặc theo sequence.

```text
payload.video_id
payload.frame_id
payload.timestamp
```

CSV:

```text
L10_V001,1200,1850,2100,2450
```

<div style="background:#fffbeb;border-left:6px solid #f59e0b;padding:14px;border-radius:10px;color:#78350f">
<b>Điểm backend cần nhớ:</b> Dù model có trả đường dẫn frame khác, cuối cùng vẫn phải map được về <code>video_id</code> và <code>frame_id</code> đúng theo quy chuẩn BTC.
</div>

## 17. Gợi ý metadata backend nên trả để frontend càng hữu ích

| Field | Type | Dùng để làm gì trên frontend? |
|---|---|---|
| `score` | number | Hiển thị độ liên quan, sort kết quả |
| `payload.video_id` | string | Card, preview, CSV |
| `payload.frame_id` | string/number | Card, preview, CSV |
| `payload.timestamp` | number | Mở video đúng thời điểm, TRAKE |
| `payload.img_link` | string | Thumbnail/preview |
| `payload.caption` | string | Overview trong modal |
| `payload.ocr` | string | Tab OCR, filter OCR |
| `payload.asr` | string | Speech transcript, filter ASR |
| `payload.objects` | string[] | YOLO tags/filter |
| `payload.people_count` | number | People counting filter |
| `payload.model` | string | Debug model nào trả kết quả |
| `payload.collection` | string | Debug collection/index |
| `payload.event_index` | number | Nâng cấp TRAKE |
| `payload.answer` | string | Gợi ý QA answer |
| `payload.explanation` | string | Giải thích vì sao match, nếu dùng VLM/RAG |

## 18. Kế hoạch tích hợp từng giai đoạn

### Phase 1 — Frontend mock chạy độc lập

- Dùng [src/data/mockResults.ts](src/data/mockResults.ts).
- [src/api.ts](src/api.ts) trả mock result.
- Mục tiêu: UI, recheck, CSV, preview chạy ổn.

### Phase 2 — Backend search API giả lập

- Backend dựng `/api/search` trả data format giống mock.
- Chưa cần model thật.
- Frontend sửa [src/api.ts](src/api.ts) để gọi backend.
- Mục tiêu: confirm CORS/proxy/API contract.

### Phase 3 — Nối vector DB thật

- Backend build index offline.
- `/api/search` dùng vector DB trả topK.
- Frontend không đổi nhiều, chỉ test result có ảnh/metadata đúng.

### Phase 4 — Nối YOLO/OCR/ASR/filter

- Backend trả objects, people_count, ocr, asr.
- Frontend gửi filter từ SearchPanel.
- PreviewModal hiển thị metadata rõ hơn.

### Phase 5 — QA/TRAKE nâng cao

- Backend thêm answer suggestion cho QA.
- Backend thêm temporal alignment/event candidates cho TRAKE.
- Frontend có thể nâng cấp Answer Builder nếu cần.

## 19. Checklist API trước khi bàn giao cho frontend

Backend team nên kiểm tra các mục này trước khi báo frontend nối:

- [ ] `GET /api/health` trả OK.
- [ ] `POST /api/search` nhận query, top_k, model_id, collection.
- [ ] Response có `results` là array.
- [ ] Mỗi result có `payload.video_id`.
- [ ] Mỗi result có `payload.frame_id`.
- [ ] Mỗi result có ảnh mở được qua `payload.img_link` hoặc `image_path`.
- [ ] `score` là number, không phải string lộn xộn.
- [ ] Top K trả không vượt quá request quá nhiều.
- [ ] Ảnh load được từ máy frontend, không chỉ từ máy backend.
- [ ] CORS hoặc Vite proxy đã xử lý.
- [ ] Khi chạy LAN, không hard-code `localhost` sai chỗ.
- [ ] Nếu có QA, trả/gợi ý answer field rõ ràng.
- [ ] Nếu có TRAKE, frame có timestamp hoặc frame_id có thể sắp xếp thời gian.

## 20. Checklist frontend cần làm khi backend sẵn sàng

Frontend phụ trách giao diện chỉ cần làm các bước sau:

1. Mở [src/api.ts](src/api.ts).
2. Thay mock search bằng request tới `/api/search` hoặc backend URL thật.
3. Nếu backend trả field khác, map response về format `SearchResult` trong [src/store.ts](src/store.ts).
4. Mở [src/utils/media.ts](src/utils/media.ts) để kiểm tra ảnh/video có URL đúng chưa.
5. Chạy frontend bằng `npm run dev`.
6. Test query đơn giản.
7. Bấm preview từng card.
8. Add vài item sang RecheckPanel.
9. Bấm CSV và kiểm tra file nộp đúng KIS/QA/TRAKE.
10. Test trên máy khác qua LAN nếu thi/team cần dùng chung.

## 21. Debug lỗi thường gặp khi nối backend

| Triệu chứng | Khả năng lỗi | Kiểm tra ở đâu? |
|---|---|---|
| Bấm Search không ra gì | API lỗi hoặc response sai format | DevTools Network, [src/api.ts](src/api.ts) |
| Có result nhưng ảnh không hiện | `img_link` sai hoặc media endpoint lỗi | [src/utils/media.ts](src/utils/media.ts), mở URL ảnh trực tiếp |
| CSV sai video/frame | Backend trả sai `payload.video_id`/`payload.frame_id` | Response `/api/search` |
| Người khác vào UI được nhưng Search lỗi | Backend URL hard-code localhost | [vite.config.ts](vite.config.ts), [src/api.ts](src/api.ts) |
| CORS error | Frontend gọi domain khác backend chưa allow | Backend CORS hoặc Vite proxy |
| Mode QA/TRAKE không đúng | Frontend mode chưa gửi hoặc backend chưa xử lý | [SearchPanel.tsx](src/components/SearchPanel.tsx), [src/api.ts](src/api.ts) |
| TopK không đúng | Backend ignore `top_k` | Backend search service |
| Result bị chậm/lag | Ảnh quá nặng hoặc trả quá nhiều | Media compression, topK, lazy loading |
| Preview không đủ thông tin | Backend chưa trả caption/OCR/ASR/object | Response payload |

## 22. Cách backend ghi tài liệu ngược lại cho frontend

Khi backend hoàn thành một phần, hãy gửi frontend 1 đoạn handoff ngắn theo mẫu này:

```text
API Base URL:
- Local: http://localhost:8000
- LAN: http://192.168.x.x:8000

Search endpoint:
- POST /api/search

Request fields:
- query, mode, top_k, collection, model_id, language, filters

Response fields:
- id, score, image_path, payload.video_id, payload.frame_id, payload.img_link, payload.timestamp, payload.caption, payload.ocr, payload.asr, payload.objects

Media endpoint:
- GET /api/media/image/{path}
- GET /api/media/video/{video_id}

Known limitations:
- QA answer chưa auto-fill
- TRAKE mới trả frame candidates, chưa group event
```

Càng thống nhất sớm, frontend càng ít phải sửa.

## 23. Phần nào của frontend nên giữ ổn định?

| File | Có nên để backend sửa không? | Lý do |
|---|---:|---|
| [src/App.tsx](src/App.tsx) | Không nên | Layout chính, sửa dễ vỡ giao diện |
| [src/store.ts](src/store.ts) | Chỉ sửa khi cần field mới | Là state trung tâm |
| [src/api.ts](src/api.ts) | Có thể sửa cùng frontend | Đây là adapter API chính |
| [src/utils/media.ts](src/utils/media.ts) | Có thể sửa cùng frontend | Liên quan media URL backend |
| [SearchPanel.tsx](src/components/SearchPanel.tsx) | Chỉ sửa khi thêm input/filter | UI query/filter |
| [ResultCard.tsx](src/components/ResultCard.tsx) | Chỉ sửa khi thêm metadata hiển thị | Card result |
| [RecheckPanel.tsx](src/components/RecheckPanel.tsx) | Cẩn thận khi sửa | Liên quan CSV submit format |
| [PreviewModal.tsx](src/components/PreviewModal.tsx) | Có thể mở rộng tab | Hiển thị metadata/video/QA |

<div style="background:#f8fafc;border:1px solid #94a3b8;padding:14px;border-radius:10px;color:#0f172a">
<b>Quy tắc team:</b> Backend muốn đổi field response thì báo frontend để map ở <code>api.ts</code>, không nên bắt mọi component đổi theo field backend thô.
</div>

## 24. Tóm tắt cực ngắn cho backend team

<div style="background:#0f172a;color:white;padding:18px;border-radius:14px;border:1px solid #334155">
  <h3 style="margin-top:0;color:#93c5fd">Backend chỉ cần nhớ 5 việc</h3>
  <ol style="line-height:1.8">
    <li>Không bỏ model AI vào <code>src/</code> frontend. Hãy đặt ở <code>backend/</code> hoặc repo backend riêng.</li>
    <li>Dựng API chính <code>POST /api/search</code>.</li>
    <li>Trả response có <code>results</code>, mỗi item có <code>payload.video_id</code>, <code>payload.frame_id</code>, <code>payload.img_link</code>, <code>score</code>.</li>
    <li>Serve ảnh/video qua endpoint hoặc URL mà browser mở được.</li>
    <li>Khi backend thật sẵn sàng, frontend gần như chỉ sửa <a style="color:#bfdbfe" href="src/api.ts"><code>src/api.ts</code></a> và <a style="color:#bfdbfe" href="src/utils/media.ts"><code>src/utils/media.ts</code></a>.</li>
  </ol>
</div>

## 25. Trang link nhanh

### Frontend files

- [src/App.tsx](src/App.tsx) — layout 3 cột và header.
- [src/store.ts](src/store.ts) — state trung tâm và type dữ liệu frontend.
- [src/api.ts](src/api.ts) — điểm nối backend chính.
- [src/utils/media.ts](src/utils/media.ts) — đổi path ảnh/video thành URL.
- [src/components/SearchPanel.tsx](src/components/SearchPanel.tsx) — query, mode, topK, filter.
- [src/components/ResultGrid.tsx](src/components/ResultGrid.tsx) — grid kết quả.
- [src/components/ResultCard.tsx](src/components/ResultCard.tsx) — card frame.
- [src/components/PreviewModal.tsx](src/components/PreviewModal.tsx) — preview lớn.
- [src/components/RecheckPanel.tsx](src/components/RecheckPanel.tsx) — Answer Builder và CSV.
- [src/components/VideoModal.tsx](src/components/VideoModal.tsx) — xem video nếu backend có media video.
- [src/data/mockResults.ts](src/data/mockResults.ts) — mock data, sau này không dùng khi backend thật.
- [vite.config.ts](vite.config.ts) — cấu hình proxy/host/port.
- [package.json](package.json) — scripts chạy frontend.

### Backend handoff cần gửi frontend

- API base URL.
- Endpoint search.
- Request format.
- Response format.
- Media URL format.
- Danh sách model/collection.
- Cách chạy backend local/LAN.
- Các limitation hiện tại.